<a href="https://colab.research.google.com/github/vamsiporeddy123/CSA6102-DIgital-Forencics/blob/main/Exp_32.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from datetime import datetime

TS_FMT = "%Y-%m-%d %H:%M:%S"

def detect_timestomping(file_meta, change_gap_minutes=60):
    """
    Detect possible timestomping based on file timestamps.

    file_meta contains:
    - born
    - modified
    - accessed
    - changed
    """

    m = datetime.strptime(file_meta["modified"], TS_FMT)
    a = datetime.strptime(file_meta["accessed"], TS_FMT)
    c = datetime.strptime(file_meta["changed"], TS_FMT)
    b = datetime.strptime(file_meta["born"], TS_FMT)

    reasons = []

    if m < b:
        reasons.append("Modified time is earlier than Born (creation) time.")

    if a < b:
        reasons.append("Accessed time is earlier than Born (creation) time.")

    gap_minutes = abs((c - m).total_seconds()) / 60

    if gap_minutes > change_gap_minutes and c > m:
        reasons.append(
            f"MFT Changed time is {gap_minutes:.0f} minutes after Modified time. "
            "Metadata may have been altered after the file was created."
        )

    return len(reasons) > 0, reasons


normal_file = {
    "born": "2026-01-10 09:00:00",
    "modified": "2026-01-10 09:05:00",
    "accessed": "2026-01-12 14:00:00",
    "changed": "2026-01-10 09:05:00",
}

tampered_file = {
    "born": "2026-02-01 12:00:00",
    "modified": "2020-01-01 00:00:00",
    "accessed": "2026-02-01 12:00:00",
    "changed": "2026-02-01 12:03:00",
}


print("=" * 60)
print("DIGITAL FORENSICS - TIMESTOMPING ANALYSIS")
print("=" * 60)

for name, file in [("Normal File", normal_file), ("Tampered File", tampered_file)]:

    suspicious, reasons = detect_timestomping(file)

    print(f"\n{name}")
    print("-" * 60)
    print("Born      :", file["born"])
    print("Modified  :", file["modified"])
    print("Accessed  :", file["accessed"])
    print("Changed   :", file["changed"])

    if suspicious:
        print("\nStatus : SUSPICIOUS")
        print("Reasons:")
        for reason in reasons:
            print(" -", reason)
    else:
        print("\nStatus : NORMAL")
        print("No evidence of timestomping detected.")

print("\n" + "=" * 60)
print("RUNNING TEST CASES")
print("=" * 60)

is_susp1, reasons1 = detect_timestomping(normal_file)
assert is_susp1 is False

is_susp2, reasons2 = detect_timestomping(tampered_file)
assert is_susp2 is True
assert any("Modified time is earlier" in r for r in reasons2)

print("✓ Test Case 1 Passed")
print("✓ Test Case 2 Passed")
print("✓ Test Case 3 Passed")

print("\nAll test cases passed successfully.")

DIGITAL FORENSICS - TIMESTOMPING ANALYSIS

Normal File
------------------------------------------------------------
Born      : 2026-01-10 09:00:00
Modified  : 2026-01-10 09:05:00
Accessed  : 2026-01-12 14:00:00
Changed   : 2026-01-10 09:05:00

Status : NORMAL
No evidence of timestomping detected.

Tampered File
------------------------------------------------------------
Born      : 2026-02-01 12:00:00
Modified  : 2020-01-01 00:00:00
Accessed  : 2026-02-01 12:00:00
Changed   : 2026-02-01 12:03:00

Status : SUSPICIOUS
Reasons:
 - Modified time is earlier than Born (creation) time.
 - MFT Changed time is 3201843 minutes after Modified time. Metadata may have been altered after the file was created.

RUNNING TEST CASES
✓ Test Case 1 Passed
✓ Test Case 2 Passed
✓ Test Case 3 Passed

All test cases passed successfully.
